# DWBI Week 3 — Fact & Dimension Tables
**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A**

PakFreight's flat `Shipments_Unified` (34 rows, 578,650 kg) becomes **one fact table + five dimensions** — and the totals do not move by a single kilogram.

## T1 — Fact, dimension, degenerate or drop?

Twelve columns the operations team asked for. Verdicts (workbook `W3_Star_Team.xlsx`, sheet T1):

| # | Column | Verdict | Lives in | Why |
|---|---|---|---|---|
| 1 | Driver_Name | **D** | DimDriver (new) | Descriptive context about the driver, stored once — not a number |
| 2 | Days_Late | **F** | FactShipment | Count of late days; additive (total late-days is meaningful) |
| 3 | Invoice_Number | **DD** | FactShipment | Operational reference with no dimension behind it — classic degenerate |
| 4 | Discount_Percent | **F** ⚠️ | FactShipment | TRICK: a rate that *looks* summable — never SUM a percent; store discount PKR |
| 5 | Truck_Registration_No | **D** | DimVehicle (new) | TRICK: looks degenerate, but a truck has type/capacity/depot — a real entity |
| 6 | Container_Count | **F** | FactShipment | Plain additive count |
| 7 | Customer_Credit_Rating | **D** | DimCustomer (future) | Attribute of the customer, not of the shipment event |
| 8 | Cost_Per_Tonne_KM | **F** ⚠️ | FactShipment | TRICK: a ratio — store cost and tonne-km, DIVIDE the sums |
| 9 | Row_Inserted_Timestamp | **X** | ETL audit log | Load metadata, not business reality — not a dimensional column at all |
| 10 | Fuel_Surcharge_PKR | **F** | FactShipment | Additive money |
| 11 | Weather_Condition | **D** | DimShipment (junk) | Descriptive context, never a measure |
| 12 | Is_Hazardous_Cargo | **D** | DimShipment (junk) | A flag describing the cargo |

**Defence Q1** — which X would an analyst fight over? `Row_Inserted_Timestamp`: "we need auditability!"
My defence: audit lives in ETL metadata tables; the dimensional model answers business questions,
and a load timestamp on every fact row answers none of them.

**Defence Q2** — one D deserving its own table: `Truck_Registration_No` → **DimVehicle**
(VehicleType, CapacityKG, HomeDepot). A registration with no attributes stays degenerate;
a truck with a type, a capacity and a depot is an entity.

**Defence Q3** — Days_Late is additive (SUM = total late-days); the trap is *averaging* it —
an average of a per-day ratio repeats the T3 CostPerKG error.

In [1]:
# T1 rule of thumb, checked against our own model
print("Fact   = numbers that happened (WeightKG, CostPKR) -> SUM them")
print("Dim    = text that describes (City, Mode, Status)  -> GROUP BY them")
print("DD     = operational IDs with no table (ShipmentID) -> keep, don't join")
print("X      = ETL metadata (Row_Inserted_Timestamp)      -> audit log, not the model")

Fact   = numbers that happened (WeightKG, CostPKR) -> SUM them
Dim    = text that describes (City, Mode, Status)  -> GROUP BY them
DD     = operational IDs with no table (ShipmentID) -> keep, don't join
X      = ETL metadata (Row_Inserted_Timestamp)      -> audit log, not the model
